# KI-Labor: Gedächtnis und Kontext mit Qwen
In diesem Experiment untersuchen wir, wie ein Sprachmodell Informationen verarbeitet. Wir nutzen das Modell **Qwen2.5**, das ohne Anmeldung frei verfügbar ist.

**Hinweis:** Das Modell ist ca. 3,1 GB groß.

---

## Was du nach diesem Notebook weißt:
- Warum ein KI-Chatbot sich normalerweise **nicht** an frühere Nachrichten erinnert
- Was ein **Kontextfenster** ist und wie es sich von menschlichem Gedächtnis unterscheidet
- Wie man durch einen Gesprächsverlauf ein "Kurzzeitgedächtnis" **simuliert**
- Was **System-Prompts** sind und wie sie das Verhalten der KI steuern

In [ ]:
# Installation der notwendigen Bibliotheken

In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install transformers accelerate

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline
import torch

# pipeline() lädt das Modell und bereitet es für die Textgenerierung vor.
# dtype bestimmt die Rechengenauigkeit: bfloat16 (GPU) spart Speicher, float32 (CPU) ist kompatibler.
# device_map='auto' wählt automatisch GPU oder CPU je nach verfügbarer Hardware.
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

---

## Teil 1: Das Experiment "Gedächtnistest"
Wir senden zwei **getrennte** Nachrichten an die KI. Da wir beim zweiten Mal den Gesprächsverlauf nicht mitschicken, erhält das Modell keinerlei Information aus der ersten Nachricht, es ist, als würde man eine völlig neue Unterhaltung beginnen.

In [ ]:
# 1. Nachricht: Wir stellen uns vor
chat1 = [{"role": "user", "content": "Hi, my name is Max!"}]
out1 = pipe(chat1, max_new_tokens=50)  # max_new_tokens begrenzt die Länge der Antwort (in Tokens, ca. 0,75 Wörter pro Token)
print("KI Antwort 1:", out1[0]['generated_text'][-1]['content'])

print("\n--- NEUE ANFRAGE (kein Verlauf Ã¼bergeben) ---\n")

# 2. Nachricht: Wir fragen nach unserem Namen, ohne den vorherigen Verlauf
chat2 = [{"role": "user", "content": "What is my name?"}]
out2 = pipe(chat2, max_new_tokens=50)
print("KI Antwort 2:", out2[0]['generated_text'][-1]['content'])

### Reflexion, bevor du weitermachst:
Beantworte diese Fragen kurz für dich (oder diskutiere sie mit deiner Sitznachbarin / deinem Sitznachbarn):

1. Warum kennt die KI den Namen nicht mehr? Liegt das an einem "Fehler" des Modells?
2. Wie läuft das bei ChatGPT oder anderen Chatbots ab, haben die wirklich ein "Gedächtnis"?
3. Was müsste man technisch ändern, damit die KI den Namen doch noch kennt?

---

## Teil 2: Die Lösung, Gedächtnis-Simulation über den Gesprächsverlauf
Sprachmodelle haben kein echtes Gedächtnis. Sie verarbeiten immer nur den Text, der ihnen in diesem Moment übergeben wird, den sogenannten **Kontext**. Wenn wir also wollen, dass die KI "weiß", was vorher gesagt wurde, müssen wir ihr den gesamten bisherigen Gesprächsverlauf als Liste mitgeben.

In [ ]:
# Der gesamte GesprÃ¤chsverlauf wird als Liste übergeben.
# Jede Nachricht hat eine 'role' (user oder assistant) und einen 'content' (den Text).
verlauf = [
    {"role": "user",      "content": "Hi, my name is Max and I love sports!"},
    {"role": "assistant", "content": "Hello Max! It is nice to meet a sports enthusiast. What kind of sports do you like?"},
    {"role": "user",      "content": "What is my name and what do I like?"}
]

output = pipe(verlauf, max_new_tokens=50)
print("KI Antwort mit Geä¤chtnis:")
print(output[0]['generated_text'][-1]['content'])

---

## Übung:

**Schritt 1 (Pflicht):** Ändere den `verlauf` so ab, dass das Sprachmodell eine bestimmte Rolle einnimmt (z.B. *a grumpy Bavarian*, *a robot from the future* oder *a funny pirate*). Füge dazu ganz oben in der Liste ein:
```python
{"role": "system", "content": "You are a grumpy Bavarian."}
```

**Schritt 2 (Erweiterung):** Verlängere das Gespräch im `verlauf` um mindestens 2 weitere Nachrichten. Wie verhält sich die KI dabei? Bleibt sie in ihrer Rolle?

**Reflexion:** Was sagt das Experiment über die Möglichkeiten und Grenzen dieses "Gedächtnisses" aus? Was passiert wohl, wenn ein Gespräch sehr lang wird?

In [ ]:
# Hier ist Platz für deine Lösung:
verlauf_uebung = [
    # Füge hier deine system-Rolle und dein Gespräch ein
]

output_uebung = pipe(verlauf_uebung, max_new_tokens=100)
print(output_uebung[0]['generated_text'][-1]['content'])

---

## Fazit

Was wir heute gesehen haben, gilt für **alle** großen Sprachmodelle, auch für ChatGPT, Gemini oder Copilot:

- Sie haben **kein echtes Langzeitgedächtnis**. Bei jeder Anfrage erhält das Modell den gesamten bisherigen Gesprächsverlauf als Texteingabe, das ist das sogenannte **Kontextfenster**.
- Dieses Kontextfenster ist **begrenzt**: Bei älteren Modellen auf einige Tausend, bei modernen auf Millionen von Tokens. Wenn das Fenster voll ist, fällt älterer Gesprächsverlauf heraus, die KI wird "vergesslicher".
- **System-Prompts** bestimmen, wie sich die KI verhält, welche Rolle sie einnimmt, welchen Ton sie verwendet oder was sie nicht tun soll. Jeder Dienst, der KI einsetzt, nutzt solche Prompts im Hintergrund.

Das erklärt, warum Chatbots manchmal inkonsistent wirken: Es gibt kein Bewusstsein und kein Gedächtnis, nur Textverarbeitung auf Basis dessen, was im aktuellen Kontext steht.